main flow

client -> gateway 
            -> schema validation -> httpx client request -> ingestion service -> returns data with status

In [ ]:
pip install -r /home/rik07/Documents/repos/simple_rag/services/gateway/requirements.txt

In [ ]:
pip freeze > /home/rik07/Documents/repos/simple_rag/services/gateway/requirements.txt

pydantic schema

In [ ]:
# from pydantic import BaseModel, Field
# from fastapi import UploadFile, File

# class IngestionSchema(BaseModel):
#     pdf_file: str = Field(default_factory=str)

connection with ingestion service

In [ ]:
from fastapi import UploadFile
import httpx

async def connecting_2_ingestion(payload: dict):
    INGESTION_URL = "http://127.0.0.1:8001/ingest"

    
    async with httpx.AsyncClient(follow_redirects=True) as client:
        return await client.post(
            url=INGESTION_URL, 
            files=payload,
            timeout=10
        )

    if response.status_code != 201:
        print(f"Error! status: {response.status_code}")
        print(f"Raw body: {response.text}")
    
    try:
        return response.json(), response.status_code
    except Exception as exp:
        raise HTTPException(f"Failed to decode JSON from: {INGESTION_URL} {exp}")
         

defining the gateway router

In [ ]:
from fastapi import FastAPI, HTTPException, status


app = FastAPI()
@app.post(path="/ingest")
async def handling_ingestion(uploaded_data: UploadFile):
    try:
        uploaded_data_content = await uploaded_data.read()
        payload = {
            "uploaded_data": (uploaded_data.file, uploaded_data_content, uploaded_data.content_type)
        }

        response = await connecting_2_ingestion(payload=payload)
        
        return {
            "data": response.text,
            "status_code": response.status_code
        }
    except Exception as exp:
        raise HTTPException (
            status_code=status.HTTP_404_NOT_FOUND,
            detail="exception occured"
        )

starting point

In [ ]:
import nest_asyncio, uvicorn, asyncio

nest_asyncio.apply()

if __name__ == "__main__":
    # setting the uvicorn configuration
    config = uvicorn.Config(
        app=app,
        host="127.0.0.1",
        port=8000,
        loop="asyncio"
    )
    # instead of using uvicorn.run(), .Server is used
    server = uvicorn.Server(config=config)

    loop = asyncio.get_event_loop() # to make sure uvicorn use the same server (localhost) as jupyter notebook
    loop.create_task(server.serve())
